In [1]:
import cv2
import torch
from ultralytics import YOLO

# =========================================
# CONFIGURAÇÃO DO JOGO
# =========================================
# Mapeie aqui os nomes EXATOS das classes do seu modelo
# Ex: se seu modelo retorna 'pedra', mude para: 'rock': 'pedra'
CLASS_MAP = {
    'rock': 'rock',         # ou 'pedra'
    'paper': 'paper',       # ou 'papel'
    'scissors': 'scissors'  # ou 'tesoura'
}

# Configurações de exibição
font = cv2.FONT_HERSHEY_SIMPLEX
COLOR_P1 = (255, 200, 0)  # Azul Ciano
COLOR_P2 = (0, 0, 255)    # Vermelho
COLOR_TEXT = (255, 255, 255)

# Placar Global
score_p1 = 0
score_p2 = 0

# Estado do Jogo
# "playing": Procurando mãos
# "result": Mostrando resultado e esperando o 'OK'
game_state = "playing"
last_result_text = ""
winner_detected = None # 'p1', 'p2', ou 'draw'

# =========================================
# FUNÇÕES AUXILIARES
# =========================================

def determine_winner(p1_move, p2_move):
    """Retorna 'p1', 'p2' ou 'draw' baseado nas regras clássicas."""
    # Normalizar para letras minúsculas para evitar erros
    p1 = p1_move.lower()
    p2 = p2_move.lower()

    # Mapeamento reverso para garantir comparação (caso o modelo use portugues)
    # Aqui assumimos que p1 e p2 já são as strings detectadas pelo modelo
    
    if p1 == p2:
        return "draw"
    
    # Regras de Vitória do P1
    # Rock ganha de Scissors
    # Scissors ganha de Paper
    # Paper ganha de Rock
    if (CLASS_MAP['rock'] in p1 and CLASS_MAP['scissors'] in p2) or \
       (CLASS_MAP['scissors'] in p1 and CLASS_MAP['paper'] in p2) or \
       (CLASS_MAP['paper'] in p1 and CLASS_MAP['rock'] in p2):
        return "p1"
    
    return "p2"

# =========================================
# INICIALIZAÇÃO
# =========================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Dispositivo em uso: {device.upper()}")

# Carregar modelo
model = YOLO("runs/detect/yolo_simples_treino/weights/best.pt")
model.to(device)

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    raise IOError("❌ Não foi possível acessar a câmera.")

# Definir resolução (opcional, ajuda a fixar o centro)
cap.set(3, 1280)
cap.set(4, 720)

print("🎥 Controles:")
print("   - Mostre as duas mãos para jogar.")
print("   - Pressione 'ESPAÇO' para confirmar o resultado (Botão OK).")
print("   - Pressione 'q' para sair.\n")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Espelhar a câmera para ficar intuitivo (esquerda é esquerda)
    frame = cv2.flip(frame, 1)
    
    h, w, _ = frame.shape
    center_x = w // 2

    # ---------------------------------------------------------
    # ESTADO 1: JOGANDO (Detectando mãos)
    # ---------------------------------------------------------
    if game_state == "playing":
        # Desenhar linha divisória
        cv2.line(frame, (center_x, 0), (center_x, h), (200, 200, 200), 2)
        cv2.putText(frame, "PLAYER 1", (50, 50), font, 1, COLOR_P1, 2)
        cv2.putText(frame, "PLAYER 2", (w - 200, 50), font, 1, COLOR_P2, 2)

        # Inferência
        results = model.predict(frame, imgsz=640, device=device, verbose=False, conf=0.5)
        
        # Variáveis para guardar o que foi detectado neste frame
        p1_hand = None
        p2_hand = None

        # Processar detecções
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0]
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)
            
            cls_id = int(box.cls[0])
            label_name = model.names[cls_id]
            conf = float(box.conf[0])

            # Desenhar Box
            color = COLOR_P1 if cx < center_x else COLOR_P2
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
            cv2.putText(frame, f"{label_name} {conf:.1f}", (int(x1), int(y1)-10), font, 0.6, color, 2)

            # Lógica de atribuição (Esquerda vs Direita)
            if cx < center_x:
                p1_hand = label_name
            else:
                p2_hand = label_name

        # Checar se temos as duas mãos para decidir o vencedor
        if p1_hand and p2_hand:
            winner = determine_winner(p1_hand, p2_hand)
            
            if winner == "p1":
                score_p1 += 1
                last_result_text = "PLAYER 1 WINS!"
                winner_detected = "p1"
            elif winner == "p2":
                score_p2 += 1
                last_result_text = "PLAYER 2 WINS!"
                winner_detected = "p2"
            else:
                last_result_text = "DRAW!"
                winner_detected = "draw"
            
            # Mudar estado para resultado
            game_state = "result"

    # ---------------------------------------------------------
    # ESTADO 2: RESULTADO (Esperando "OK")
    # ---------------------------------------------------------
    elif game_state == "result":
        # Efeito visual de "overlay" escuro
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w, h), (0, 0, 0), -1)
        frame = cv2.addWeighted(overlay, 0.7, frame, 0.3, 0)

        # Mostrar vencedor
        color_res = COLOR_TEXT
        if winner_detected == "p1": color_res = COLOR_P1
        elif winner_detected == "p2": color_res = COLOR_P2
        
        # Centralizar texto
        text_size = cv2.getTextSize(last_result_text, font, 2, 3)[0]
        text_x = (w - text_size[0]) // 2
        cv2.putText(frame, last_result_text, (text_x, h//2 - 20), font, 2, color_res, 3)

        # Botão Simulado "OK"
        btn_text = "[ PRESS SPACE TO CONTINUE ]"
        cv2.putText(frame, btn_text, ((w//2)-200, h//2 + 80), font, 0.8, (255, 255, 255), 2)

    # ---------------------------------------------------------
    # INTERFACE FIXA (Placar)
    # ---------------------------------------------------------
    # Mostrar Placar sempre
    cv2.rectangle(frame, (center_x - 100, h - 60), (center_x + 100, h - 10), (50, 50, 50), -1)
    score_str = f"{score_p1} x {score_p2}"
    
    # Centralizar placar
    s_size = cv2.getTextSize(score_str, font, 1.2, 2)[0]
    cv2.putText(frame, score_str, (center_x - s_size[0]//2, h - 20), font, 1.2, (255, 255, 255), 2)

    # Mostrar Janela
    cv2.imshow("YOLOv8 - Duel Mode", frame)

    # Controles de Teclado
    key = cv2.waitKey(1) & 0xFF
    
    if key == ord("q"):
        break
    
    # Se estiver na tela de resultado e apertar ESPAÇO, reseta para jogar
    if game_state == "result" and key == 32: # 32 é Espaço
        game_state = "playing"

cap.release()
cv2.destroyAllWindows()
print("🛑 Execução encerrada.")

✅ Dispositivo em uso: CUDA
🎥 Controles:
   - Mostre as duas mãos para jogar.
   - Pressione 'ESPAÇO' para confirmar o resultado (Botão OK).
   - Pressione 'q' para sair.

🛑 Execução encerrada.
